In [7]:
!pip install -qU sarvamai qdrant-client langchain-qdrant langchain-huggingface \
    langchain-text-splitters langchain-groq langchain-community pymupdf \
    requests beautifulsoup4 tqdm ipywidgets

In [8]:
import os

DOWNLOAD_DIR   = "acts_pdfs"
TEXT_DIR       = "acts_text"
INDEX_FILE     = "all_acts_index.json"
DB_PATH        = "vidhi_vichara_db"
COLLECTION     = "indian_laws"

SCRAPE_START     = 0
SCRAPE_END       = 21701
SCRAPE_WORKERS   = 15
DOWNLOAD_WORKERS = 15

EMBED_MODEL   = "BAAI/bge-small-en-v1.5"
EMBED_DIM     = 384
CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 200
INGEST_BATCH  = 100
TOP_K         = 5

GROQ_MODEL = "llama-3.3-70b-versatile"
LLM_TEMP   = 0.0

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(TEXT_DIR, exist_ok=True)
print("Config ready. Directories created.")

Config ready. Directories created.


In [9]:
from getpass import getpass

os.environ["SARVAM_API_KEY"] = getpass("Enter Sarvam API Key: ")
os.environ["GROQ_API_KEY"]   = getpass("Enter Groq API Key: ")
print("Keys loaded!")

[WARN] handle 13550: HTTPSConnectionPool(host='www.indiacode.nic.in', port=443): Max retries exceeded with url: /handle/123456789/13550 (Caused by ResponseError('too many 500 error responses'))
[WARN] handle 13546: HTTPSConnectionPool(host='www.indiacode.nic.in', port=443): Max retries exceeded with url: /handle/123456789/13546 (Caused by ResponseError('too many 500 error responses'))
[WARN] handle 13553: HTTPSConnectionPool(host='www.indiacode.nic.in', port=443): Max retries exceeded with url: /handle/123456789/13553 (Caused by ResponseError('too many 500 error responses'))
[WARN] handle 13547: HTTPSConnectionPool(host='www.indiacode.nic.in', port=443): Max retries exceeded with url: /handle/123456789/13547 (Caused by ResponseError('too many 500 error responses'))
[WARN] handle 13558: HTTPSConnectionPool(host='www.indiacode.nic.in', port=443): Max retries exceeded with url: /handle/123456789/13558 (Caused by ResponseError('too many 500 error responses'))
[WARN] handle 13548: HTTPSConn

Enter Sarvam API Key:  ········
Enter Groq API Key:  ········


Keys loaded!


In [10]:
import requests
import json
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import time

BASE    = "https://www.indiacode.nic.in"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

def make_session():
    """Session with retry backoff to avoid IP bans."""
    s = requests.Session()
    retry = Retry(
        total=3,
        backoff_factor=1.5,
        status_forcelist=[429, 500, 502, 503, 504],
    )
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.headers.update(HEADERS)
    return s

def get_act_info(handle_id):
    url     = f"{BASE}/handle/123456789/{handle_id}"
    session = make_session()
    try:
        r = session.get(url, timeout=15)
        if r.status_code != 200:
            return handle_id, None
        soup    = BeautifulSoup(r.text, "html.parser")
        pdf_url = None
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if "/bitstream/123456789/" in href and ".pdf" in href.lower():
                pdf_url = BASE + href if href.startswith("/") else href
                break
        if not pdf_url:
            return handle_id, None
        title_tag = soup.find("title")
        name = (
            title_tag.get_text(strip=True).replace("India Code:", "").strip()
            if title_tag
            else f"Act_{handle_id}"
        )
        return handle_id, {"handle_id": handle_id, "name": name, "url": url, "pdf_url": pdf_url}
    except Exception as e:
        print(f"[WARN] handle {handle_id}: {e}")
        return handle_id, None

all_ids  = range(SCRAPE_START, SCRAPE_END)
all_acts = []

with ThreadPoolExecutor(max_workers=SCRAPE_WORKERS) as executor:
    futures = {executor.submit(get_act_info, hid): hid for hid in all_ids}
    for future in tqdm(as_completed(futures), total=len(all_ids), desc="Scanning India Code"):
        hid, act = future.result()
        if act:
            all_acts.append(act)

with open(INDEX_FILE, "w") as f:
    json.dump(all_acts, f, indent=2)

print(f"Total acts found: {len(all_acts)}")

Scanning India Code:   9%|▊         | 1850/21701 [02:17<51:54,  6.37it/s]  

[WARN] handle 17292: HTTPSConnectionPool(host='www.indiacode.nic.in', port=443): Max retries exceeded with url: /handle/123456789/17292 (Caused by ResponseError('too many 500 error responses'))


Scanning India Code:   9%|▊         | 1851/21701 [02:27<8:31:06,  1.54s/it]

[WARN] handle 1850: HTTPSConnectionPool(host='www.indiacode.nic.in', port=443): Max retries exceeded with url: /handle/123456789/1850 (Caused by ResponseError('too many 500 error responses'))
[WARN] handle 17296: HTTPSConnectionPool(host='www.indiacode.nic.in', port=443): Max retries exceeded with url: /handle/123456789/17296 (Caused by ResponseError('too many 500 error responses'))


Scanning India Code:   9%|▊         | 1852/21701 [02:27<7:26:14,  1.35s/it]

...

[WARN] handle 7070: HTTPSConnectionPool(host='www.indiacode.nic.in', port=443): Max retries exceeded with url: /handle/123456789/7070 (Caused by ResponseError('too many 500 error responses'))


Scanning India Code:  33%|███▎      | 7077/21701 [19:37<1:00:42,  4.01it/s]

[WARN] handle 7075: HTTPSConnectionPool(host='www.indiacode.nic.in', port=443): Max retries exceeded with url: /handle/123456789/7075 (Caused by ResponseError('too many 500 error responses'))
[WARN] handle 7076: HTTPSConnectionPool(host='www.indiacode.nic.in', port=443): Max retries exceeded with url: /handle/123456789/7076 (Caused by ResponseError('too many 500 error responses'))
[WARN] handle 7071: HTTPSConnectionPool(host='www.indiacode.nic.in', port=443): Max retries exceeded with url: /handle/123456789/7071 (Caused by ResponseError('too many 500 error responses'))


Scanning India Code: 100%|██████████| 21701/21701 [31:22<00:00, 11.53it/s] 


Total acts found: 1906


In [11]:
import threading

with open(INDEX_FILE) as f:
    all_acts = json.load(f)

failed = []
lock   = threading.Lock()

def download_one(act):
    filename       = f"{DOWNLOAD_DIR}/act_{act['handle_id']}.pdf"
    act["local_path"] = filename
    if os.path.exists(filename) and os.path.getsize(filename) > 500:
        return act, True
    session = make_session()
    try:
        r = session.get(act["pdf_url"], timeout=60, stream=True)
        r.raise_for_status()
        with open(filename, "wb") as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)
        return act, True
    except Exception as e:
        return act, str(e)

print(f"Downloading {len(all_acts)} PDFs with {DOWNLOAD_WORKERS} workers...")

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    futures = {executor.submit(download_one, act): act for act in all_acts}
    with tqdm(total=len(all_acts), desc="Downloading PDFs") as pbar:
        for future in as_completed(futures):
            act, result = future.result()
            if result is not True:
                with lock:
                    failed.append({"name": act["name"], "error": result})
            pbar.update(1)

with open(INDEX_FILE, "w") as f:
    json.dump(all_acts, f, indent=2)

downloaded = sum(1 for a in all_acts if a.get("local_path") and os.path.exists(a["local_path"]))
print(f"\nDownloaded: {downloaded} | Failed: {len(failed)}")
if failed:
    print("Sample failures:", failed[:3])


Downloaded: 1733 | Failed: 173
Sample failures: [{'name': 'Jharkhand Special Court Act, 2016', 'error': '404 Client Error: Not Found for url: https://www.indiacode.nic.in/bitstream/123456789/3052/1/Jharkhand%20Special%20Court%20Act%2C%202016.pdf'}, {'name': 'Assam State Higher Education Council Act, 2017', 'error': "Invalid URL '/bitstream/123456789/1618/2/A1974-52.pdf': No scheme supplied. Perhaps you meant https:///bitstream/123456789/1618/2/A1974-52.pdf?"}, {'name': 'Assam Agricultural Income Tax (Amendment) Act, 2004', 'error': '404 Client Error: Not Found for url: https://www.indiacode.nic.in/bitstream/123456789/3088/1/The%20Assam%20Agricultural%20Income%20Tax%20%28Amendment%29%20Act%2C%202004..pdf'}]


In [12]:
import fitz  

with open(INDEX_FILE) as f:
    all_acts = json.load(f)

scanned_pdfs    = []
extracted_count = 0

for act in tqdm(all_acts, desc="Extracting Text"):
    path = act.get("local_path")
    if not path or not os.path.exists(path):
        continue

    text_path = f"{TEXT_DIR}/act_{act['handle_id']}.txt"

    if os.path.exists(text_path):
        act["text_path"] = text_path
        extracted_count += 1
        continue

    try:
        doc  = fitz.open(path)
        text = "\n".join(page.get_text() for page in doc)
        doc.close()
        if len(text.strip()) > 300:
            with open(text_path, "w", encoding="utf-8") as f:
                f.write(text)
            act["text_path"] = text_path
            extracted_count += 1
        else:
            scanned_pdfs.append(act)
    except Exception as e:
        print(f"[WARN] fitz failed for {act.get('name', act['handle_id'])}: {e}")
        scanned_pdfs.append(act)

with open(INDEX_FILE, "w") as f:
    json.dump(all_acts, f, indent=2)
with open("scanned_pdfs.json", "w") as f:
    json.dump(scanned_pdfs, f, indent=2)

print(f"Extracted: {extracted_count} | Scanned/Failed (need Sarvam OCR): {len(scanned_pdfs)}")

Extracting Text:  37%|███▋      | 702/1906 [00:43<00:22, 53.44it/s]

MuPDF error: format error: non-page object in page tree



Extracting Text: 100%|██████████| 1906/1906 [02:08<00:00, 14.88it/s]

Extracted: 1614 | Scanned/Failed (need Sarvam OCR): 119


In [13]:
import zipfile
import glob
import shutil
from sarvamai import SarvamAI

with open("scanned_pdfs.json") as f:
    scanned = json.load(f)

if not scanned:
    print("No scanned PDFs to process. Skipping.")
else:
    client        = SarvamAI(api_subscription_key=os.environ["SARVAM_API_KEY"])
    success       = 0
    failed_sarvam = 0
    print(f"Processing {len(scanned)} scanned PDFs via Sarvam...")

    for act in tqdm(scanned, desc="Sarvam OCR"):
        text_path = f"{TEXT_DIR}/act_{act['handle_id']}.txt"
        pdf_path  = act.get("local_path")

        if os.path.exists(text_path) or not pdf_path or not os.path.exists(pdf_path):
            continue

        zip_out = f"sarvam_out_{act['handle_id']}.zip"
        ext_dir = f"sarvam_ext_{act['handle_id']}"

        try:
            job    = client.document_intelligence.create_job(language="en-IN", output_format="md")
            job.upload_file(pdf_path)
            job.start()
            status = job.wait_until_complete()

            if status.job_state == "Completed":
                job.download_output(zip_out)
                with zipfile.ZipFile(zip_out, "r") as z:
                    z.extractall(ext_dir)
                md_files = glob.glob(f"{ext_dir}/*.md")
                if md_files:
                    with open(md_files[0], "r", encoding="utf-8") as f:
                        text = f.read()
                    with open(text_path, "w", encoding="utf-8") as f:
                        f.write(text)
                    act["text_path"] = text_path
                    success += 1
        except Exception as e:
            print(f"[WARN] Sarvam failed for {act['name']}: {e}")
            failed_sarvam += 1
        finally:
            # Clean up temp files to avoid disk bloat
            if os.path.exists(zip_out):
                os.remove(zip_out)
            if os.path.exists(ext_dir):
                shutil.rmtree(ext_dir, ignore_errors=True)

        time.sleep(1)

    with open(INDEX_FILE) as f:
        all_acts = json.load(f)
    for act in all_acts:
        text_path = f"{TEXT_DIR}/act_{act['handle_id']}.txt"
        if os.path.exists(text_path):
            act["text_path"] = text_path
    with open(INDEX_FILE, "w") as f:
        json.dump(all_acts, f, indent=2)

    print(f"Sarvam done. Success: {success} | Failed: {failed_sarvam}")

Processing 119 scanned PDFs via Sarvam...


Sarvam OCR:   0%|          | 0/119 [00:00<?, ?it/s]

[WARN] Sarvam failed for Assam Agricultural Income Tax  Act, 1939: headers: {'date': 'Sun, 10 May 2026 19:15:16 GMT', 'content-type': 'application/json', 'content-length': '140', 'connection': 'keep-alive', 'server': 'uvicorn', 'x-request-id': '20260510_6549de2d-5f0c-46e7-bc97-16d2b5c45570', 'access-control-allow-origin': '*', 'strict-transport-security': 'max-age=31536000; includeSubDomains; preload'}, status_code: 429, body: {'error': {'message': 'No credits available.', 'code': 'insufficient_quota_error', 'request_id': '20260510_6549de2d-5f0c-46e7-bc97-16d2b5c45570'}}


Sarvam OCR:   1%|          | 1/119 [00:05<10:56,  5.56s/it]

[WARN] Sarvam failed for Assam Agricultural Income Tax (Amendment) Act, 1973: headers: {'date': 'Sun, 10 May 2026 19:15:21 GMT', 'content-type': 'application/json', 'content-length': '140', 'connection': 'keep-alive', 'server': 'uvicorn', 'x-request-id': '20260510_ffefa649-3c4b-4197-8da7-262f278e1285', 'access-control-allow-origin': '*', 'strict-transport-security': 'max-age=31536000; includeSubDomains; preload'}, status_code: 429, body: {'error': {'message': 'No credits available.', 'code': 'insufficient_quota_error', 'request_id': '20260510_ffefa649-3c4b-4197-8da7-262f278e1285'}}


...

[WARN] Sarvam failed for Madhya Pradesh Ayurvedic, Unani and Naturopathic Practitioner Act 1970: headers: {'date': 'Sun, 10 May 2026 19:24:39 GMT', 'content-type': 'application/json', 'content-length': '140', 'connection': 'keep-alive', 'server': 'uvicorn', 'x-request-id': '20260510_4b9c9378-e13a-465d-beff-098071d274ae', 'access-control-allow-origin': '*', 'strict-transport-security': 'max-age=31536000; includeSubDomains; preload'}, status_code: 429, body: {'error': {'message': 'No credits available.', 'code': 'insufficient_quota_error', 'request_id': '20260510_4b9c9378-e13a-465d-beff-098071d274ae'}}


Sarvam OCR:  99%|█████████▉| 118/119 [09:29<00:04,  4.85s/it]

[WARN] Sarvam failed for Bihar Goshala Act, 1950: headers: {'date': 'Sun, 10 May 2026 19:24:44 GMT', 'content-type': 'application/json', 'content-length': '139', 'connection': 'keep-alive', 'server': 'uvicorn', 'x-request-id': '20260510_dd105ffe-8432-4c3a-b46c-cd7db4109f7a', 'access-control-allow-origin': '*', 'strict-transport-security': 'max-age=31536000; includeSubDomains; preload'}, status_code: 429, body: {'error': {'message': 'Rate limit exceeded', 'code': 'rate_limit_exceeded_error', 'request_id': '20260510_dd105ffe-8432-4c3a-b46c-cd7db4109f7a'}}


Sarvam OCR: 100%|██████████| 119/119 [09:34<00:00,  4.82s/it]

Sarvam done. Success: 0 | Failed: 119


In [14]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


_lock_file = os.path.join(DB_PATH, ".lock")
if os.path.exists(_lock_file):
    os.remove(_lock_file)
    print("[INFO] Removed stale Qdrant lock file.")

with open(INDEX_FILE) as f:
    all_acts = json.load(f)

acts_with_text = [a for a in all_acts if a.get("text_path") and os.path.exists(a["text_path"])]
print(f"Acts with extracted text: {len(acts_with_text)}")

print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", " ", ""],
)

q_client = QdrantClient(path=DB_PATH)
try:
    collection_exists = any(c.name == COLLECTION for c in q_client.get_collections().collections)
    if not collection_exists:
        q_client.create_collection(
            collection_name=COLLECTION,
            vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
        )
        print("Qdrant collection created.")
        ingested_handles = set()
    else:
        print("Qdrant collection already exists — doing incremental update.")
        ingested_handles = set()
        offset = None
        while True:
            records, offset = q_client.scroll(
                collection_name=COLLECTION,
                limit=1000,
                offset=offset,
                with_payload=["handle_id"],
            )
            for r in records:
                ingested_handles.add(str(r.payload.get("handle_id", "")))
            if offset is None:
                break
        print(f"Already ingested handle IDs: {len(ingested_handles)}")

    try:
        point_id = q_client.count(collection_name=COLLECTION).count
    except Exception:
        point_id = 0

    new_acts = [a for a in acts_with_text if str(a["handle_id"]) not in ingested_handles]
    print(f"Acts to ingest this run: {len(new_acts)}")

    for act in tqdm(new_acts, desc="Ingesting into Qdrant"):
        try:
            with open(act["text_path"], "r", encoding="utf-8") as f:
                text = f.read()
            chunks = splitter.split_text(text)
            if not chunks:
                continue
            for i in range(0, len(chunks), INGEST_BATCH):
                batch   = chunks[i : i + INGEST_BATCH]
                vectors = embeddings.embed_documents(batch)
                points  = [
                    PointStruct(
                        id=point_id + j,
                        vector=vectors[j],
                        payload={
                            "text":      batch[j],
                            "act_name":  act["name"],
                            "handle_id": str(act["handle_id"]),
                            "url":       act["url"],
                        },
                    )
                    for j in range(len(batch))
                ]
                q_client.upsert(collection_name=COLLECTION, points=points)
                point_id += len(batch)
        except Exception as e:
            print(f"[WARN] Ingest failed for {act['name']}: {e}")

    print(f"\nDone! Total vectors in Qdrant: {point_id}")

finally:
    q_client.close()
    print("Qdrant connection closed.")

Acts with extracted text: 1614
Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Qdrant collection created.
Acts to ingest this run: 1614


Ingesting into Qdrant:  20%|█▉        | 317/1614 [04:22<12:21,  1.75it/s]  /tmp/ipykernel_57/4230294626.py:91: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20047 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  q_client.upsert(collection_name=COLLECTION, points=points)
Ingesting into Qdrant: 100%|██████████| 1614/1614 [26:14<00:00,  1.03it/s] 


Done! Total vectors in Qdrant: 111103
Qdrant connection closed.


In [17]:
import gc
import re
gc.collect()

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore

if not os.path.exists(DB_PATH):
    raise FileNotFoundError("DB not found — re-run Cell 8 first!")

_lock_file = os.path.join(DB_PATH, ".lock")
if os.path.exists(_lock_file):
    os.remove(_lock_file)
    print("[INFO] Removed stale Qdrant lock file.")

print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

print("Connecting to Qdrant...")
q_client    = QdrantClient(path=DB_PATH)
vectorstore = QdrantVectorStore(client=q_client, collection_name=COLLECTION, embedding=embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

print("Initialising Groq LLM...")
llm = ChatGroq(
    api_key=os.environ["GROQ_API_KEY"],
    model_name=GROQ_MODEL,
    temperature=LLM_TEMP,
)

system_prompt = (
    "You are an expert Indian constitutional and tax lawyer auditor. "
    "The retrieved context below comes from actual Indian parent legislation (Central Acts, State Acts). "
    "The Proposed Rule is an executive action such as a circular, notification, or guideline. "
    "Your task is to compare the Proposed Rule against the retrieved Established Law. "
    "Identify any instances where the Proposed Rule oversteps, deviates, or contradicts the established law. "
    "A score of 1-3 means the rule is perfectly aligned or is a minor clarification. "
    "A score of 4-6 means there is a procedural deviation but not a rights violation. "
    "A score of 7-10 means the rule contradicts fundamental rights or core legislative intent. "
    "CRITICAL INSTRUCTION: Any alteration, extension, or contradiction of statutory timelines (e.g., days, months), numerical limits, or basic worker/citizen rights MUST be flagged as a major deviation (Score 7+), regardless of exceptions or general intent. "
    "You MUST output your response strictly in the following JSON format and nothing else. "
    "Do NOT add any text before or after the JSON.\n\n"
    "{{\n"
    '  "deviation_score": <integer between 1 and 10>,\n'
    '  "relevant_act": "Name of the specific Indian Act that is most relevant",\n'
    '  "location_of_deviation": "What specific part of the proposed rule is violating the law",\n'
    '  "explanation": "Brief explanation of the clash",\n'
    '  "suggested_fix": "How to rephrase the proposed rule to align with the established law"\n'
    "}}\n\n"
    "Established Law/Context:\n{context}"
)


prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Proposed Rule: {input}"),
])

def format_docs(docs):
    return "\n\n".join(
        f"[{doc.metadata.get('act_name', 'Unknown')}]\n{doc.page_content}"
        for doc in docs
    )

def safe_parse_score(raw_score):
    """Handles '7', '7 - Major deviation', or plain int from LLM."""
    match = re.search(r"\d+", str(raw_score))
    return int(match.group()) if match else 0

def score_to_signal(score: int) -> str:
    if score <= 3:
        return "🟢 GREEN — Well aligned"
    elif score <= 6:
        return "🟡 AMBER — Tweaks needed"
    else:
        return "🔴 RED — Major deviation"

rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Vidhi-Vichara Engine is ready!")

[INFO] Removed stale Qdrant lock file.
Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Connecting to Qdrant...


/tmp/ipykernel_57/2493108773.py:26: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <indian_laws> contains 111103 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  q_client    = QdrantClient(path=DB_PATH)


Initialising Groq LLM...
Vidhi-Vichara Engine is ready!


In [18]:
import json

test_rule = "All taxpayer applications for Form 10A must be submitted electronically by 30.04.2024."

print(f"Testing Rule:\n  '{test_rule}'\n")
print("Analysing for deviations...\n")

answer = rag_chain.invoke(test_rule)

print("─── VIDHI-VICHARA REPORT ───\n")
try:
    parsed = json.loads(answer)
    score  = safe_parse_score(parsed["deviation_score"])
    print(f"SIGNAL: {score_to_signal(score)}\n")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError:
    print("[WARN] Could not parse JSON. Raw output:\n")
    print(answer)

Testing Rule:
  'All taxpayer applications for Form 10A must be submitted electronically by 30.04.2024.'

Analysing for deviations...

─── VIDHI-VICHARA REPORT ───

SIGNAL: 🔴 RED — Major deviation

{
  "deviation_score": 7,
  "relevant_act": "Income-tax Act, 1961",
  "location_of_deviation": "Specific deadline of 30.04.2024 for electronic submission",
  "explanation": "The proposed rule may contradict the Income-tax Act, 1961, and related rules which typically provide for a due date for filing of applications and returns, often with provisions for extensions. The Act and rules may not specify a particular date like 30.04.2024 for electronic submission of Form 10A, potentially altering statutory timelines.",
  "suggested_fix": "Rephrase the proposed rule to align with existing deadlines and procedures under the Income-tax Act, 1961, and related rules, allowing for flexibility in submission methods and timelines as per the Act and rules."
}


In [19]:
spectrum_tests = [
    {
        "name": "Perfect Alignment (Expected: 1-3)",
        "rule": (
            "Every public limited company must hold an Annual General Meeting within six months "
            "from the close of the financial year and provide at least 21 days clear notice to its members."
        ),
    },
    {
        "name": "Procedural Slip (Expected: 4-6)",
        "rule": (
            "Under the Right to Information protocol, public authorities have a maximum of 60 days "
            "to respond to a standard citizen request for information."
        ),
    },
    {
        "name": "Rights Violation (Expected: 7-9)",
        "rule": (
            "Private corporations are legally permitted to terminate the employment of any female staff "
            "member if she requires more than 12 weeks of maternity leave."
        ),
    },
    {
        "name": "Constitutional Crisis (Expected: 10)",
        "rule": (
            "State police forces are authorised to detain suspects in custody for up to 14 days without "
            "producing them before a judicial magistrate, provided the arresting officer signs a written justification."
        ),
    },
]

print("Initiating Vidhi-Vichara Spectrum Calibration...\n" + "=" * 70)

for test in spectrum_tests:
    print(f"\nTEST: {test['name']}")
    print(f"Rule: '{test['rule']}'")
    print("Analysing...\n")

    result = rag_chain.invoke(test["rule"])

    try:
        parsed = json.loads(result)
        score  = safe_parse_score(parsed["deviation_score"])
        print(f"SIGNAL: {score_to_signal(score)} | Score: {score}/10")
        print(json.dumps(parsed, indent=2))
    except json.JSONDecodeError:
        print("[WARN] Raw output:")
        print(result)

    print("\n" + "=" * 70)

Initiating Vidhi-Vichara Spectrum Calibration...

TEST: Perfect Alignment (Expected: 1-3)
Rule: 'Every public limited company must hold an Annual General Meeting within six months from the close of the financial year and provide at least 21 days clear notice to its members.'
Analysing...

SIGNAL: 🟢 GREEN — Well aligned | Score: 1/10
{
  "deviation_score": 1,
  "relevant_act": "Companies Act, 2013",
  "location_of_deviation": "None",
  "explanation": "The proposed rule aligns with Section 96 of the Companies Act, 2013, which requires every public company to hold an Annual General Meeting within six months from the close of the financial year and provide at least 21 days clear notice to its members.",
  "suggested_fix": "No changes are required as the proposed rule is in line with the established law."
}


TEST: Procedural Slip (Expected: 4-6)
Rule: 'Under the Right to Information protocol, public authorities have a maximum of 60 days to respond to a standard citizen request for informat

In [20]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

title_html = HTML(
    "<h3 style='font-family:monospace;color:#2c3e50'>⚖️ Vidhi-Vichara — Live Rule Auditor</h3>"
    "<p style='color:grey'>Enter any proposed rule, circular, or notification below and click <b>Audit</b>.</p>"
)

text_area = widgets.Textarea(
    placeholder="e.g. All taxpayer applications for Form 10A must be submitted electronically by 30.04.2024.",
    layout=widgets.Layout(width="95%", height="100px"),
)
audit_btn = widgets.Button(description="⚖️ Audit", button_style="primary", layout=widgets.Layout(width="120px"))
clear_btn = widgets.Button(description="Clear",    button_style="warning",  layout=widgets.Layout(width="80px"))
out       = widgets.Output()

def on_audit(b):
    rule = text_area.value.strip()
    if not rule:
        return
    with out:
        clear_output()
        print("Analysing...")
        result = rag_chain.invoke(rule)
        try:
            parsed = json.loads(result)
            score  = safe_parse_score(parsed["deviation_score"])
            clear_output()
            print(f"\nSIGNAL: {score_to_signal(score)}  |  Score: {score}/10\n")
            print(json.dumps(parsed, indent=2))
        except json.JSONDecodeError:
            clear_output()
            print("[WARN] Could not parse JSON. Raw output:\n")
            print(result)

def on_clear(b):
    text_area.value = ""
    with out:
        clear_output()

audit_btn.on_click(on_audit)
clear_btn.on_click(on_clear)

display(title_html, text_area, widgets.HBox([audit_btn, clear_btn]), out)

Textarea(value='', layout=Layout(height='100px', width='95%'), placeholder='e.g. All taxpayer applications for…

Output()